# SIDRA no Databricks: população e saneamento

Vamos consultar as **sete tabelas usadas pelo pipeline SINAN–IBGE** e guardar
cada uma em um **DataFrame Spark**. O objetivo desta etapa é conhecer as fontes,
as consultas e as colunas retornadas. As tabelas continuam separadas, com os
valores originais do SIDRA.

| Tabela | Conteúdo | Período | DataFrame |
| --- | --- | --- | --- |
| [136](https://sidra.ibge.gov.br/tabela/136) | População residente do Censo | 2010 | `df136` |
| [4709](https://sidra.ibge.gov.br/tabela/4709) | População residente do Censo | 2022 | `df4709` |
| [6579](https://sidra.ibge.gov.br/tabela/6579) | Estimativas populacionais | Anos selecionados abaixo | `df6579` |
| [3218](https://sidra.ibge.gov.br/tabela/3218) | Água, esgotamento, lixo e energia elétrica | 2010 | `df3218` |
| [6803](https://sidra.ibge.gov.br/tabela/6803) | Abastecimento de água | 2022 | `df6803` |
| [6805](https://sidra.ibge.gov.br/tabela/6805) | Esgotamento sanitário | 2022 | `df6805` |
| [6892](https://sidra.ibge.gov.br/tabela/6892) | Destino do lixo | 2022 | `df6892` |

Os recortes seguem `utils/analysis_pipeline.py`, mas este notebook é independente
dos módulos e dos arquivos locais do projeto. Cada execução consulta o SIDRA ao vivo.

## Preparação

Importe este `.ipynb` no Databricks, conecte-o a um compute com Python/Spark e
execute as células na ordem. O Databricks fornece `spark` e `display`.
Instalamos apenas o cliente HTTP; PySpark já faz parte do ambiente.
O compute precisa conseguir acessar `https://apisidra.ibge.gov.br`.

In [0]:
catalog = "dengue_sus"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

In [0]:
%pip install httpx==0.28.1

In [0]:
import httpx
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

SIDRA_API = "https://apisidra.ibge.gov.br/values"
MUNICIPIOS = "all"  # n6: todos os municípios. Para experimentar: "3550308" (São Paulo).
TAMANHO_LOTE = 7

## Como ler uma consulta

Exemplo: `/t/136/n6/all/v/93/p/2010/c86/0/h/y`.

| Trecho | Significado |
| --- | --- |
| `t/136` | Tabela 136 |
| `n6/all` | Todos os municípios (nível territorial 6) |
| `v/93` | Variável 93: população residente |
| `p/2010` | Período 2010 |
| `c86/0` | Classificação cor ou raça, categoria Total |
| `h/y` | Incluir uma primeira linha com os nomes das colunas |

`all` em uma classificação solicita todas as suas categorias; `0` seleciona
Total. As variáveis de população retornam pessoas, enquanto as variáveis de
saneamento escolhidas aqui retornam percentuais. Leia também as colunas
`Variável` e `Unidade de Medida` de cada resposta.

A função abaixo faz uma requisição, separa o cabeçalho e cria um DataFrame.
A requisição acontece no processo Python do notebook; os dados retornados são
então entregues ao Spark. Os códigos, `Valor` e os marcadores especiais do SIDRA
permanecem como texto. Converter valores ou interpretar ausências fica para uma
próxima etapa.

In [0]:
def consultar_sidra(url):
    """Lê uma resposta SIDRA com cabeçalho e preserva seus valores como texto."""
    print(f"Consultando: {url}")
    try:
        resposta = httpx.get(url, timeout=180, follow_redirects=True)
        resposta.raise_for_status()
        linhas = resposta.json()
        if not isinstance(linhas, list) or len(linhas) < 2:
            raise ValueError("Resposta sem registros ou em formato inesperado")
        if not all(isinstance(linha, dict) for linha in linhas):
            raise ValueError("Cabeçalho ou registros inválidos")

        cabecalho = linhas[0]
        if cabecalho.get("V") != "Valor":
            raise ValueError("Cabeçalho SIDRA ausente; use /h/y na URL")
        chaves = list(cabecalho)
        if any(set(linha) != set(chaves) for linha in linhas[1:]):
            raise ValueError("Colunas dos registros diferentes do cabeçalho")
    except (httpx.HTTPError, ValueError) as erro:
        raise RuntimeError(f"Falha na consulta SIDRA: {url} — {erro}") from erro

    esquema = StructType([
        StructField(cabecalho[chave], StringType(), True) for chave in chaves
    ])
    registros = [tuple(linha[chave] for chave in chaves) for linha in linhas[1:]]
    return spark.createDataFrame(registros, schema=esquema)

In [0]:
import re
import unicodedata
from pyspark.sql import DataFrame
def normalize_column_names(df:DataFrame) ->DataFrame:
    def normalize(col:str) -> str:
        name = unicodedata.normalize("NFKD",col)
        name = "".join(c for c in name if not unicodedata.combining(c))
        name = name.lower().translate(str.maketrans("","","()"))
        return re.sub(r"[^a-z0-9]+", "_", name).strip("_")

    names = [normalize(col) for col in df.columns]

    if len(names) != len(set(names)):
        raise ValueError("A normalização gerou nomes  de colunas duplicados")

    return df.toDF(*names)
    

## 1. População do Censo 2010 — tabela 136

Usamos a variável **93** e a categoria **Total de cor ou raça (`c86/0`)**.
O resultado tem uma linha por município e será uma fonte de denominadores
populacionais na futura integração.

In [0]:
url136 = f"{SIDRA_API}/t/136/n6/{MUNICIPIOS}/v/93/p/2010/c86/0/h/y"
df136 = consultar_sidra(url136)

In [0]:
print("df136:", df136.count(), "linhas")
df136.printSchema()
display(df136.limit(10))

In [0]:
df136 = normalize_column_names(df136)
df136.printSchema()

In [0]:
df136.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog}.bronze.2010_populacao_censo")

## 2. População do Censo 2022 — tabela 4709

Também selecionamos a variável **93**, agora no período **2022**.
Esta consulta não precisa da classificação de cor ou raça usada na tabela 136.

In [0]:
url4709 = f"{SIDRA_API}/t/4709/n6/{MUNICIPIOS}/v/93/p/2022/h/y"
df4709 = consultar_sidra(url4709)

In [0]:
print("df4709:", df4709.count(), "linhas")
df4709.printSchema()
display(df4709.limit(10))

In [0]:
df4709 =normalize_column_names(df4709)
df4709.printSchema()

In [0]:
df4709.write.format("delta").mode("overwrite").option("mergeSchema",'true').saveAsTable(f"{catalog}.bronze.2022_populacao_censo")

## 3. Estimativas anuais de população — tabela 6579

A variável **9324** fornece a população residente estimada. Vamos usar os mesmos
anos históricos do pipeline: **2001–2006, 2008–2009 e 2011–2021**. Esta lista é
um recorte do projeto, não uma lista de todos os períodos disponíveis no SIDRA.
As estimativas de 2024–2025 do notebook anterior ficam fora deste recorte.

Consultamos até sete anos por requisição para manter as respostas menores.
`unionByName` empilha os lotes alinhando as colunas pelo nome: não é um join.
O resultado tem uma linha por município e ano.

In [0]:
ANOS_ESTIMATIVAS = [
    2001, 2002, 2003, 2004, 2005, 2006, 2008, 2009,
    2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
]

lotes6579 = []
for inicio in range(0, len(ANOS_ESTIMATIVAS), TAMANHO_LOTE):
    anos = ANOS_ESTIMATIVAS[inicio:inicio + TAMANHO_LOTE]
    periodos = ",".join(map(str, anos))
    url6579 = f"{SIDRA_API}/t/6579/n6/{MUNICIPIOS}/v/9324/p/{periodos}/h/y"
    lotes6579.append(consultar_sidra(url6579))

df6579 = lotes6579[0]
for lote in lotes6579[1:]:
    df6579 = df6579.unionByName(lote)

In [0]:
print("df6579:", df6579.count(), "linhas")
df6579.printSchema()
display(df6579.limit(10))

In [0]:
df6579 = normalize_column_names(df6579 )
df6579.printSchema()

In [0]:
df6579.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.populacao_residente_estimada")

## 4. Condições dos domicílios em 2010 — tabela 3218

A variável **1000096** representa o percentual de domicílios particulares
permanentes. Fazemos quatro consultas: **água (`c61`), esgotamento (`c299`),
lixo (`c67`) e energia (`c309`)**. Em cada consulta, uma classificação varia e
as outras ficam em Total (`0`), como nas requisições do pipeline.

Pedimos todas as categorias de cada dimensão, inclusive Total. O DataFrame
preserva essas linhas brutas; a seleção dos indicadores feita pelo pipeline
não é aplicada nesta etapa. Não some categorias junto com seus totais.
A coluna adicional `dimensao` identifica a consulta de origem. Portanto,
**há várias linhas por município** e os totais das quatro consultas são mantidos.

In [0]:
CLASSIFICACOES_3218 = {
    "agua": "c61/all/c299/0/c67/0/c309/0",
    "esgotamento": "c61/0/c299/all/c67/0/c309/0",
    "lixo": "c61/0/c299/0/c67/all/c309/0",
    "energia": "c61/0/c299/0/c67/0/c309/all",
}

quadros3218 = []
for dimensao, classificacoes in CLASSIFICACOES_3218.items():
    url3218 = (
        f"{SIDRA_API}/t/3218/n6/{MUNICIPIOS}/v/1000096/p/2010/"
        f"{classificacoes}/h/y"
    )
    quadro = consultar_sidra(url3218).withColumn("dimensao", F.lit(dimensao))
    quadros3218.append(quadro)

df3218 = quadros3218[0]
for quadro in quadros3218[1:]:
    df3218 = df3218.unionByName(quadro)

In [0]:
print("df3218:", df3218.count(), "linhas")
df3218.printSchema()
display(df3218.limit(10))

In [0]:
df3218 = normalize_column_names(df3218)
df3218.printSchema()

In [0]:
df3218.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.2010_senamento")

## 5. Abastecimento de água em 2022 — tabela 6803

Usamos a variável **1000381**, percentual de domicílios particulares permanentes
ocupados, e a classificação **1821** (ligação à rede geral e principal forma de
abastecimento). A lista abaixo reproduz as categorias usadas pelo pipeline.
As descrições de cada categoria vêm na resposta do SIDRA.

As categorias são consultadas em lotes de até sete e reunidas em `df6803`.
Cada município aparece em várias linhas, uma por categoria solicitada.

In [0]:
CATEGORIAS_AGUA_2022 = [
    "72144", "72146", "72147", "72148", "72149", "72150", "72151",
    "72152", "72154", "72155", "72156", "72157", "72158", "72159", "72160",
]

lotes6803 = []
for inicio in range(0, len(CATEGORIAS_AGUA_2022), TAMANHO_LOTE):
    categorias = ",".join(CATEGORIAS_AGUA_2022[inicio:inicio + TAMANHO_LOTE])
    url6803 = (
        f"{SIDRA_API}/t/6803/n6/{MUNICIPIOS}/v/1000381/p/2022/"
        f"c1821/{categorias}/h/y"
    )
    lotes6803.append(consultar_sidra(url6803))

df6803 = lotes6803[0]
for lote in lotes6803[1:]:
    df6803 = df6803.unionByName(lote)

In [0]:
print("df6803:", df6803.count(), "linhas")
df6803.printSchema()
display(df6803.limit(10))

In [0]:
df6803= normalize_column_names(df6803)
df6803.printSchema()

In [0]:
df6803.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.2022_abastecimento_agua")

## 6. Esgotamento sanitário em 2022 — tabela 6805

Mantemos a variável percentual **1000381** e usamos a classificação **11558**
(tipo de esgotamento sanitário). As oito categorias do pipeline incluem a
situação sem banheiro ou sanitário. Elas são divididas em dois lotes.

In [0]:
CATEGORIAS_ESGOTO_2022 = [
    "72110", "72111", "72112", "72113", "92858", "72114", "72115", "92861",
]

lotes6805 = []
for inicio in range(0, len(CATEGORIAS_ESGOTO_2022), TAMANHO_LOTE):
    categorias = ",".join(CATEGORIAS_ESGOTO_2022[inicio:inicio + TAMANHO_LOTE])
    url6805 = (
        f"{SIDRA_API}/t/6805/n6/{MUNICIPIOS}/v/1000381/p/2022/"
        f"c11558/{categorias}/h/y"
    )
    lotes6805.append(consultar_sidra(url6805))

df6805 = lotes6805[0]
for lote in lotes6805[1:]:
    df6805 = df6805.unionByName(lote)

In [0]:
print("df6805:", df6805.count(), "linhas")
df6805.printSchema()
display(df6805.limit(10))

In [0]:
df6805= normalize_column_names(df6805)
df6805.printSchema()

In [0]:
df6805.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.2022_tipo_de_esgotamento")

## 7. Destino do lixo em 2022 — tabela 6892

Usamos a variável **1000381** e a classificação **67** (destino do lixo).
As seis categorias utilizadas pelo pipeline cabem em uma requisição.

In [0]:
CATEGORIAS_LIXO_2022 = ["72120", "72121", "72122", "72123", "72124", "1091"]
categorias = ",".join(CATEGORIAS_LIXO_2022)
url6892 = (
    f"{SIDRA_API}/t/6892/n6/{MUNICIPIOS}/v/1000381/p/2022/"
    f"c67/{categorias}/h/y"
)
df6892 = consultar_sidra(url6892)

In [0]:
print("df6892:", df6892.count(), "linhas")
df6892.printSchema()
display(df6892.limit(10))

In [0]:
df6892 = normalize_column_names(df6892)
df6892.printSchema()

In [0]:
df6892.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.2022_destino_do_lixo")

## Conferência e próximo passo

Ao terminar, devem existir `df136`, `df4709`, `df6579`, `df3218`, `df6803`,
`df6805` e `df6892`, cada um com linhas e com as colunas originais do SIDRA.
Nas prévias, confira `Ano`, `Variável`, `Unidade de Medida` e as classificações.
O cabeçalho deve aparecer como nomes de colunas, nunca como uma linha de dados.

Para validar primeiro uma consulta pequena, altere `MUNICIPIOS` para `"3550308"`
e execute tudo. Depois restaure `"all"` e execute novamente para carregar o Brasil.
As prévias mostram até dez linhas; `count()` informa o total do DataFrame.

Se uma requisição falhar, a mensagem mostra sua URL. Confira o acesso do compute
ao SIDRA e os parâmetros indicados antes de reexecutar a célula.

O próximo passo será compreender os códigos e marcadores de `Valor` para então
escolher conversões e formas de integração. Nesta etapa os sete DataFrames
permanecem em memória; não há escrita de tabelas, arquivos ou indicadores.